# Tutorial: Ex4 Exponential Diffusion

This notebook gives a guided example for training and postprocessing the SDENF model on **Example 4: Exponential Diffusion**.

The notebook is meant to be a tutorial wrapper around the repository scripts. It does not reimplement the model. Instead, it shows how to:

- inspect the data, scripts generating data are in the repo: https://github.com/yJesseChen/SDEDATA-v1,
- inspect the model config,
- launch training from the existing solver,
- inspect model outputs,
- rerun prediction and postprocessing from a trained model.

## 1. Task Introduction

This example follows Section 5.2.1 of:

Yuan Chen and Dongbin Xiu, *Learning stochastic flow maps with generative models*, Journal of Computational Physics, 2024.  
Paper PDF: <https://iamyuanchen.xyz/pdf/2024ChenXiu.pdf>

The target process is a one-dimensional Ito SDE with linear drift and state-dependent exponential diffusion,

$$
dX_t = -\mu X_t\,dt + \sigma \exp(-X_t^2)\,dB_t,
$$

with the parameters used in the provided config:

$$
\mu = 5, \qquad \sigma = 0.5.
$$

The goal is to learn a stochastic flow map that advances samples of the SDE over one time step. The training data contain short trajectories from random initial conditions, while the testing data contain longer trajectories used to evaluate recursive prediction.

### Set Working Folder

This notebook is stored in `SDENF-v1/tutorials/`. The next cell switches the working folder to the parent repository folder, so all paths below are relative to `SDENF-v1`.

In [ ]:
from pathlib import Path
import os

# This notebook is stored in SDENF-v1/tutorials/.
# Move one level up so all repository-relative paths work.
os.chdir(Path.cwd().resolve().parent)

print("Current folder:", Path.cwd())

### Import Packages

Import all packages used in this notebook.


In [ ]:
import json
import re
import os
import shutil
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, Image, JSON, display
from scipy.io import loadmat

### Plot Style

Use LaTeX-style labels and fonts for all figures.


In [ ]:
# Plot style: use LaTeX-like math/text rendering for all figures in this notebook.
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "mathtext.fontset": "cm",
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
})

## 2. Preparation

### Data

The data for this example can be generated from the companion data repository:

```text
SDEDATA-v1/Ex4ExpDiff.py
```

After generation, place the `.mat` files under:

```text
SDENF-v1/data/Ex4ExpDiff/
```

For this tutorial we use:

```text
data/Ex4ExpDiff/Ex4ExpDiff_train.mat
data/Ex4ExpDiff/Ex4ExpDiff_test.mat
```

The `.mat` files should contain a variable named `data`. For this 1D example, the shape convention is:

```text
[dimension, time steps, number of trajectories]
```

In [ ]:
train_path = Path("data/Ex4ExpDiff/Ex4ExpDiff_train.mat")
test_path = Path("data/Ex4ExpDiff/Ex4ExpDiff_test.mat")

train_data = loadmat(str(train_path))["data"]
test_data = loadmat(str(test_path))["data"]

print("Training data path:", train_path)
print("Training data shape:", train_data.shape)
print("Testing data path:", test_path)
print("Testing data shape:", test_data.shape)

In [ ]:
Delta = 0.01

def plot_sample_trajectories(ax, data, title, n_show=20):
    x = data[0]
    t = np.arange(x.shape[0]) * Delta
    idx = np.linspace(0, x.shape[1] - 1, min(n_show, x.shape[1]), dtype=int)

    for j in idx:
        ax.plot(t, x[:, j], alpha=0.8, linewidth=1)
    ax.set_title(title)
    ax.set_xlabel(r"$t$")
    ax.set_ylabel(r"$X_t$")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
plot_sample_trajectories(axes[0], train_data, r"Training data: sample trajectories")
plot_sample_trajectories(axes[1], test_data, r"Testing data: sample trajectories")
plt.tight_layout()
plt.show()

### Config

The main config for this example is:

```text
config/Ex4Expdiff.json
```

The tutorial prints the full config below. The most commonly edited parts are the data paths, training size, batch size, number of epochs, and monitor switches.

In [ ]:
config_path = Path("config/Ex4Expdiff.json")
with open(config_path) as f:
    ex4_config = json.load(f)

print("Config file:", config_path)
display(JSON(ex4_config, expanded=False))

In the config file, the main blocks are:

- `eqn_config`: specifies the equation/example information, including the example name `eqn_name`, state dimension `dim`, time step `Delta`, and model parameters such as `mu` and `sigma`.

- `net_config`: specifies the normalizing-flow model and training setup.
  - `N_rec`: set to **2** in this example. This means each training sample contains one input state and one output state.
  - `fname`: set to **`MAF`**, which means we use a Masked Autoregressive Flow.
  - `flevel`: set to **5**, meaning that the flow uses 5 coupling/flow blocks.
  - `net_spec`: specifies the neural-network architecture, including the number of nodes, layers, and activation function.
  - `weight_decay`: set to **0** here, meaning no weight-decay regularization is used.
  - `l_rate_config`: specifies the learning-rate schedule. In this example, we use **`StepCyclic`** with the parameters listed in the config.
  - `batch_size`: set to **20000**. A large batch size helps make the Monte Carlo estimate in training more accurate.

- `dat_config`: specifies the training and testing data.
  - `TrainData_dir` and `TestData_dir`: paths to the training and testing data files.
  - `n_ea_traj`: set to **4**. The code samples 2 transition pairs from each training trajectory. With 10000 training trajectories, this gives **40000 training pairs**.
  - `N_pred`: number of testing trajectories used for prediction/evaluation.

- `monitor_config`: controls optional diagnostics plotted during training. **If a monitor has `if = false`, that diagnostic will not be generated.**
  - `traindata_hist` and `traintransin_hist`: plot the empirical shape/range of the training data and transition inputs.
  - `cond_mv`: plots conditional mean and variance diagnostics.
  - `Evameanv`: plots trajectory mean and variance diagnostics.
  - `loss`: plots the training loss history.
  - `repdf_display`: plots conditional PDF diagnostics.


## 3. Training

Training is done by calling the repository solver script. The result folder name is controlled by `--test_name`.

For this tutorial, the output folder is:

```text
results/Ex4Expdiff_tutorial/
```

Set `RUN_TRAINING = True` when you want to launch training from this notebook.

In [ ]:
RUN_TRAINING = True

test_name = "Ex4Expdiff_tutorial"
cmd = [
    sys.executable,
    "SolveNFSDE.py",
    f"--test_name={test_name}",
    "--config_path=./config/Ex4Expdiff.json",
    "--model_name=NFSDE",
]

print("Command:")
print(" ".join(cmd))

if RUN_TRAINING:
    subprocess.run(cmd, check=True)
else:
    print("Training was not launched. Set RUN_TRAINING = True to run this cell.")

## 4. Model Outputs

After training, the result folder should contain the saved config, model weights, prediction output, logs, and optional monitor figures.

Typical files are:

```text
results/<test_name>/Test_config.json
results/<test_name>/Test_model/model.pt
results/<test_name>/predict.mat
results/<test_name>/Monitor/
```

The next cell checks the result folder produced by training. It lists the saved config, model weights, prediction file, logs, and monitor figures so that you can quickly verify what the run generated.

In [ ]:
result_dir = Path("results") / test_name

if not result_dir.exists():
    print(f"Result folder does not exist yet: {result_dir}")
    print("Run the training cell first, or change test_name to an existing result folder.")
else:
    for path in sorted(result_dir.rglob("*")):
        if path.is_file():
            print(path)

The next cell displays representative monitor figures. For time-dependent diagnostics, we show the figure at initialization and the figure after training. The loss and learning-rate plots are shown side by side.

In [ ]:
result_dir = Path("results") / test_name
monitor_dir = result_dir / "Monitor"

def numeric_index(path):
    numbers = re.findall(r"\d+", path.stem)
    return int(numbers[0]) if numbers else 0

def collect_pngs(folder, patterns):
    files = []
    for pattern in patterns:
        files.extend(folder.glob(pattern))
    return sorted(set(files), key=lambda p: (numeric_index(p), p.name))

def first_and_last(files):
    if not files:
        return []
    if len(files) == 1:
        return [("available figure", files[0])]
    return [("initialization", files[0]), ("after training", files[-1])]

def notebook_image_path(path):
    # The notebook is stored in tutorials/, while result paths are relative to the repository root.
    # HTML images are resolved relative to the notebook file, so prepend ../ here.
    return "../" + path.as_posix()

def show_image_row(items):
    if not items:
        return
    html = '<div style="display:flex; gap:16px; align-items:flex-start; flex-wrap:wrap;">'
    for label, path in items:
        html += (
            '<div style="flex:1; min-width:360px;">'
            f'<div style="font-weight:600; margin-bottom:6px;">{label}</div>'
            f'<img src="{notebook_image_path(path)}" style="max-width:100%; height:auto; border:1px solid #ddd;"/>'
            '</div>'
        )
    html += '</div>'
    display(HTML(html))

if not monitor_dir.exists():
    print(f"Monitor folder does not exist yet: {monitor_dir}")
else:
    print(f"Monitor folder: {monitor_dir}")

    # 1. dataplot
    dataplot_dir = monitor_dir / "dataplot"
    dataplot_items = []
    if dataplot_dir.exists():
        train_hist = sorted(dataplot_dir.glob("hist_Train_data*.png"))
        trans_hist = sorted(dataplot_dir.glob("hist_input_Train_data*.png"))
        if train_hist:
            dataplot_items.append(("training data", train_hist[0]))
        if trans_hist:
            dataplot_items.append(("transition inputs", trans_hist[0]))
    print("\nData plots")
    if dataplot_items:
        for label, fig_path in dataplot_items:
            print(f"{label}: {fig_path}")
        show_image_row(dataplot_items)
    else:
        print("No dataplot figures found.")

    # 2-3. loss and lr, shown side by side
    loss_items = []
    loss_dir = monitor_dir / "loss"
    if loss_dir.exists():
        loss_files = collect_pngs(loss_dir, ["loss.png"])
        lr_files = collect_pngs(loss_dir, ["lr.png"])
        if loss_files:
            loss_items.append(("loss history", loss_files[-1]))
        if lr_files:
            loss_items.append(("learning-rate schedule", lr_files[-1]))
    print("\nLoss and learning-rate monitor")
    if loss_items:
        for label, fig_path in loss_items:
            print(f"{label}: {fig_path}")
        show_image_row(loss_items)
    else:
        print("No loss/lr figures found.")

    # 4-6. time-dependent monitor figures: first and last snapshots
    time_dependent_specs = [
        ("Conditional PDF", monitor_dir / "repdfplot", ["finalpdf*.png"]),
        ("Conditional mean and variance", monitor_dir / "condmeanvar", ["cond_mvplot*.png"]),
        ("Mean and standard deviation", monitor_dir / "Eva", ["E*M*.png"]),
    ]

    print("\nFor the following time-dependent monitor figures, we show the snapshot at initialization and the snapshot after training.")
    for title, folder, patterns in time_dependent_specs:
        files = collect_pngs(folder, patterns) if folder.exists() else []
        print(f"\n{title}")
        if not files:
            print(f"No figures found in {folder}.")
            continue
        selected = first_and_last(files)
        for label, fig_path in selected:
            print(f"{label}: {fig_path}")
        show_image_row(selected)

## 5. Postprocess

This section is for the common case where a trained model and config already exist. The required files are:

```text
results/<test_name>/Test_config.json
results/<test_name>/Test_model/model.pt
```

### Prediction

`ShowTest.py` reloads the trained model and writes a new prediction file. The output is saved under a post-test subfolder:

```text
results/<test_name>/<test_case>/predict.mat
```

Set `RUN_PREDICTION = True` to run it.

In [ ]:
RUN_PREDICTION = True
post_test_case = "post_test"

cmd = [
    sys.executable,
    "ShowTest.py",
    f"--test_name={test_name}",
    "--model_name=NFSDE",
    f"--test_case={post_test_case}",
    "--test_file=./data/Ex4ExpDiff/Ex4ExpDiff_test.mat",
]

print("Command:")
print(" ".join(cmd))

if RUN_PREDICTION:
    subprocess.run(cmd, check=True)
else:
    print("Prediction was not launched. Set RUN_PREDICTION = True to run this cell.")

### Evaluation and Figures

`ShowProdcution.py` reads the trained model and prediction output, then generates selected production figures. In practice, we usually choose which products to generate by commenting or uncommenting function calls in `ShowProdcution.py`.

For this tutorial, we output the **conditional mean** and **conditional standard deviation** comparison. The other production functions are commented out in `ShowProdcution.py`.

The figures are saved as PDF files under:

```text
results/<test_name>/<test_case>/Eva/<eqn_name>_condmeanv/
```

Set `RUN_EVALUATION = True` to run it.

In [ ]:
RUN_EVALUATION = True

cmd = [
    sys.executable,
    "ShowProdcution.py",
    f"--test_name={test_name}",
    f"--test_case={post_test_case}",
    "--model_name=NFSDE",
]

print("Command:")
print(" ".join(cmd))

if RUN_EVALUATION:
    subprocess.run(cmd, check=True)
else:
    print("Evaluation was not launched. Set RUN_EVALUATION = True to run this cell.")

In [ ]:
post_dir = Path("results") / test_name / post_test_case
eva_dir = post_dir / "Eva"
print("Postprocess folder:", post_dir)
print("Evaluation folder:", eva_dir)

if eva_dir.exists():
    files = sorted(path for path in eva_dir.rglob("*") if path.is_file())
    print(f"\nFound {len(files)} files in the evaluation folder.")

    if files:
        html = "<ul>"
        for path in files:
            notebook_relative_path = "../" + path.as_posix()
            html += f'<li><a href="{notebook_relative_path}" target="_blank">{path.as_posix()}</a></li>'
        html += "</ul>"
        display(HTML(html))
else:
    print("Evaluation folder does not exist yet. Run prediction/evaluation first.")

## Notes for Adapting This Tutorial

To adapt this workflow to a new example, usually change:

- the data generation script in `SDEDATA-v1`,
- `TrainData_dir` and `TestData_dir`,
- `eqn_name` and equation parameters,
- `dim`, `Delta`, and `N_rec`,
- model size and training settings,
- monitor switches.

For a first run, keep monitor switches off until basic training and prediction run successfully.